# Suspicious Mule Accounts Classification - EDA & Preprocessing
**Role**: Senior Banking Fraud Data Scientist  
**Project**: Mule Account Detection and Classification  

## Executive Summary
Mule accounts are intermediary bank accounts used by criminals to launder illicit funds (from scams, ransomware, identity theft, etc.). Detecting them early is critical for compliance and loss prevention.  
This notebook performs an extensive Exploratory Data Analysis (EDA) on a dataset containing 9,082 account profiles and 3,925 features, focusing on:
1. Identifying data types
2. Analyzing missing values
3. Finding duplicate columns
4. Finding constant columns
5. Pinpointing highly correlated features
6. Detecting target leakage
7. Creating data cleaning recommendations
8. Proposing fraud-specific feature engineering

In [ ]:
# Setup and Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set aesthetic style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
print("Libraries loaded successfully!")

## 1. Load the Dataset
Let's load the dataset and inspect its basic shape and target column `F3924` class distribution.

In [ ]:
# Load dataset
df = pd.read_csv("DataSet.csv")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Target column 'F3924' distribution:")
dist = df['F3924'].value_counts(dropna=False)
pct = df['F3924'].value_counts(normalize=True) * 100
for val, count in dist.items():
    print(f"  Class {val}: {count} ({pct[val]:.2f}%)")

## 2. Identify Data Types
We have a massive number of features (3925). Let's group them by data type and see if any non-numeric columns exist.

In [ ]:
# Group by data type
dtype_counts = df.dtypes.value_counts()
print("Data type counts:")
print(dtype_counts)

# Identify non-numeric columns
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNon-numeric columns count: {len(non_numeric)}")
if non_numeric:
    print(f"Non-numeric columns: {non_numeric}")
    print(df[non_numeric].head())

## 3. Find Missing Values
Missing values can impact model performance. Let's analyze missing value counts and identify columns with high missing rates (>50% and 100% missing).

In [ ]:
# Missing value analysis
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100

print(f"Total cells in dataset: {df.size}")
print(f"Total missing cells: {null_counts.sum()} ({null_counts.sum()/df.size*100:.2f}%)")
print(f"Columns with >50% missing values: {(null_pct > 50).sum()}")
print(f"Columns with 100% missing values: {(null_pct == 100).sum()}")

# Top 20 columns with missing values
print("\nTop 20 columns with highest missing value %:")
print(null_pct.sort_values(ascending=False).head(20))

## 4. Find Constant Columns
Constant columns contain zero variance (the same value for all rows). They provide zero predictive power and can be safely dropped.

In [ ]:
# Constant columns (nunique <= 1)
constant_cols = [col for col in df.columns if df[col].nunique(dropna=True) <= 1]
print(f"Found {len(constant_cols)} constant columns.")
if constant_cols:
    print(f"Sample of constant columns (first 20): {constant_cols[:20]}")

## 5. Find Duplicate Columns
Duplicate columns have identical values across all rows. They inflate model complexity without adding new information. We detect them using column hashing.

In [ ]:
# Fill NAs placeholder to compute hashes accurately
df_filled = df.fillna(-99999)

hashes = {}
duplicate_groups = []
duplicate_cols_list = []

for col in df.columns:
    try:
        col_hash = hash(tuple(df_filled[col].values))
        if col_hash in hashes:
            hashes[col_hash].append(col)
        else:
            hashes[col_hash] = [col]
    except Exception as e:
        pass
        
for h, cols in hashes.items():
    if len(cols) > 1:
        duplicate_groups.append(cols)
        duplicate_cols_list.extend(cols[1:])
        
print(f"Found {len(duplicate_cols_list)} duplicate columns.")
if duplicate_groups:
    print("\nFirst 5 duplicate groups (first column is kept, others can be dropped):")
    for g in duplicate_groups[:5]:
        print(f"  - Group: {g}")

## 6. Highly Correlated Features
Highly correlated features (r > 0.95) are redundant and cause collinearity. Let's find pairs of highly correlated features after filtering out constant and duplicate columns.

In [ ]:
# Filter columns for correlation
exclude_cols = set(constant_cols) | set(null_pct[null_pct == 100].index) | set(duplicate_cols_list)
cols_to_corr = [c for c in df.columns if c not in exclude_cols and np.issubdtype(df[c].dtype, np.number)]

print(f"Computing correlation on {len(cols_to_corr)} columns...")
df_numeric = df[cols_to_corr].fillna(df[cols_to_corr].median())
corr_matrix = df_numeric.corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.95
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
print(f"Found {len(to_drop)} features to drop due to correlation > 0.95.")

# Show top 10 correlated pairs
high_corr_pairs = []
for col in upper.columns:
    correlated_with = upper.index[upper[col] > 0.95].tolist()
    for c_with in correlated_with:
        high_corr_pairs.append((c_with, col, upper.loc[c_with, col]))

high_corr_pairs = sorted(high_corr_pairs, key=lambda x: x[2], reverse=True)
print("\nTop 10 correlated pairs:")
for f1, f2, val in high_corr_pairs[:10]:
    print(f"  - {f1} <-> {f2}: correlation = {val:.4f}")

## 7. Target Leakage Detection
Target leakage occurs when a feature contains information about the target that wouldn't be available at the time of prediction (e.g. fraud flags, post-event logs). Features with absolute correlation > 0.8 with the target should be carefully audited.

In [ ]:
# Target correlation
target_col = "F3924"
corr_with_target = df_numeric.corrwith(df[target_col]).abs()
top_target_corr = corr_with_target.sort_values(ascending=False)

leakage_candidates = top_target_corr[top_target_corr > 0.8].index.tolist()
leakage_candidates = [c for c in leakage_candidates if c != target_col]

print(f"Found {len(leakage_candidates)} leakage candidates with correlation > 0.8 with target '{target_col}':")
for col in leakage_candidates:
    print(f"  - Feature '{col}': correlation = {top_target_corr[col]:.4f}")

## 8. Visualizations
Let's visualize the target class imbalance and some key data stats.

In [ ]:
# Target Class Imbalance Plot
plt.figure(figsize=(8, 5))
sns.barplot(x=dist.index, y=dist.values, palette="viridis")
plt.title("Target Class Imbalance ('F3924')")
plt.xlabel("Mule Account Class (0 = Legitimate, 1 = Mule)")
plt.ylabel("Number of Accounts")
for i, val in enumerate(dist.values):
    plt.text(i, val + 100, f"{val} ({pct.values[i]:.2f}%)", ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.savefig("class_imbalance.png", dpi=300)
plt.show()

## 9. Cleaning & Feature Engineering Recommendations
### Data Cleaning Recommendations:
1. **Drop Constant Columns**: Remove all columns with 0 variance.
2. **Drop Duplicate Columns**: Remove redundant identical columns.
3. **Drop High Missing Value Columns**: Remove columns with >50% missing values (or 100% missing).
4. **Handle Missing Values**: Impute remaining missing values with Median (for robustness against outliers).
5. **Address Multicollinearity**: Remove one of each pair of highly correlated features (>0.95 correlation).

### Fraud Feature Engineering Opportunities:
- **Velocity Features**: Transaction frequency and velocity (e.g. count of transactions in last 24h, 7d, 30d).
- **Ratio Features**: Ratio of cash withdrawals to deposits, ratio of total outflow to inflow (mule accounts typically empty funds quickly).
- **Outlier Detection**: Distance to normal user behavior using Isolation Forest or local outlier factor.
- **Time-based Features**: Creation time of account to first transaction delay (mule accounts are often activated/used immediately after creation).